In [5]:
from datasets import load_dataset

# Using the correct namespace path for the AG News dataset
dataset = load_dataset("fancyzhx/ag_news")
print(dataset)

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [6]:
import pandas as pd

# Convert the first 5 rows of the train split to a pandas DataFrame and display it
df_train = pd.DataFrame(dataset['train'][:5])
display(df_train)

,text,label
0,Wall St. Bears Claw Back Into the Black (Reute...,2
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2
3,Iraq Halts Oil Exports from Main Southern Pipe...,2
4,"Oil prices soar to all-time record, posing new...",2


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 1. Initialize the Vectorizer
# max_features=5000 limits the vocabulary to the top 5000 most meaningful words.
# This prevents Colab from crashing due to RAM limits on massive datasets.
vectorizer = TfidfVectorizer(max_features=5000)

# 2. Fit and Transform the text
# 'Fit' calculates the IDF across all articles.
# 'Transform' calculates the TF for each specific article and builds the matrix.
# Using 'df_train' and 'text' instead of the undefined 'df_sample' and 'cleaned_text'
tfidf_matrix = vectorizer.fit_transform(df_train['text'])

# 3. Visualize the mathematical output
# Extract the vocabulary list (the column names of our new matrix)
feature_names = vectorizer.get_feature_names_out()

# Convert the matrix into a readable DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

print(f"Matrix Shape: {tfidf_df.shape[0]} articles (rows) by {tfidf_df.shape[1]} unique words (columns)")

# Let's look at the mathematical weights for the very first article
print("\nTop weighted words in Article 0:")
first_article_weights = tfidf_df.iloc[0]
print(first_article_weights[first_article_weights > 0].sort_values(ascending=False).head())

Matrix Shape: 5 articles (rows) by 122 unique words (columns)

Top weighted words in Article 0:
wall       0.420740
reuters    0.237037
band       0.210370
back       0.210370
again      0.210370
Name: 0, dtype: float64


In [12]:
import pandas as pd
import re
from datasets import load_dataset
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

# 1. Ensure NLTK resources are downloaded
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Load dataset using the correct namespace to prevent URI-parsing bugs
dataset = load_dataset("fancyzhx/ag_news")
df = pd.DataFrame(dataset['train']).head(10).copy()

# 2. Initialize the NLP tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

# 3. Define the dual-processing function
def process_text_both_ways(raw_text):
    # Phase 1: Clean the string
    text = str(raw_text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'[^a-z\s]', '', text)

    # Phase 2: Tokenize
    tokens = word_tokenize(text)

    # Phase 3: Remove stopwords
    meaningful_tokens = [word for word in tokens if word not in stop_words and len(word) > 1]

    # Phase 4: Branch into Stemming and Lemmatization
    stemmed_tokens = [stemmer.stem(word) for word in meaningful_tokens]
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in meaningful_tokens]

    # Join back into strings for DataFrame storage
    return " ".join(stemmed_tokens), " ".join(lemmatized_tokens)

# 4. Apply the function and split the results into two new columns
print("Applying stemming and lemmatization...")
df[['stemmed_text', 'lemmatized_text']] = df['text'].apply(
    lambda x: pd.Series(process_text_both_ways(x))
)

# 5. Compare the output
pd.set_option('display.max_colwidth', 100)
comparison_df = df[['text', 'stemmed_text', 'lemmatized_text']]

print("\n--- SIDE-BY-SIDE COMPARISON ---")
print(comparison_df.head(3))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Applying stemming and lemmatization...

--- SIDE-BY-SIDE COMPARISON ---
                                                                                                  text  \
0  Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindli...   
1  Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Gr...   
2  Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\abou...   

                                                                                          stemmed_text  \
0     wall st bear claw back black reuter reuter shortsel wall street dwindlingband ultracyn see green   
1  carlyl look toward commerci aerospac reuter reuter privat invest firm carlyl groupwhich reput ma...   
2  oil economi cloud stock outlook reuter reuter soar crude price plu worriesabout economi outlook ...   

                                                                              